# Comparacao dos cinco algoritmos com intervalos de confianca por bootstrap

Este notebook recalcula AUPRC, ROC-AUC e Brier Score dos cinco algoritmos ja treinados na fase de modelagem, usando as predicoes salvas para o teste de 2024.
Como o teste de 2024 tem apenas 208 obitos entre 3978 registros, qualquer estimativa pontual dessas metricas carrega incerteza relevante, que nao fica visivel numa unica tabela de valores pontuais.
Para quantificar essa incerteza, o notebook usa bootstrap, reamostrando o conjunto de teste com reposicao muitas vezes e recalculando as metricas em cada reamostragem, o que permite estimar um intervalo de confianca de 95 por cento para cada metrica de cada algoritmo.
Por fim, o notebook compara diretamente, por bootstrap pareado, os dois algoritmos com melhor AUPRC pontual, para avaliar se a diferenca entre eles e consistente ou se pode ser explicada por variacao amostral.

Importa as bibliotecas necessarias, inclui a raiz do projeto no caminho de importacao para reutilizar `utils/preprocessamento.py`, e cria os diretorios de saida usados ao longo do notebook.
A tabela final de intervalos de confianca e salva em `analysis/results_avaliacao/`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils import preprocessamento

DIRETORIO_METRICAS = PROJECT_ROOT / "output" / "metricas"
DIRETORIO_RESULTADOS_ANALISE = PROJECT_ROOT / "analysis" / "results_avaliacao"
DIRETORIO_RESULTADOS_ANALISE.mkdir(parents=True, exist_ok=True)

ALGORITMOS = ["regressao_logistica", "arvore_decisao", "random_forest", "xgboost", "lightgbm"]
NUMERO_REAMOSTRAGENS = 2000
SEMENTE_ALEATORIA = 42

Carrega, para cada um dos cinco algoritmos, as predicoes de probabilidade e o alvo real salvos pela fase de modelagem para o teste de 2024.
Confirma que o alvo real e identico entre os cinco arquivos, ja que todos vem do mesmo conjunto de teste, apenas com probabilidades preditas diferentes.

In [2]:
predicoes_teste2024 = {
    algoritmo: pd.read_parquet(DIRETORIO_METRICAS / f"{algoritmo}_predicoes_teste2024.parquet")
    for algoritmo in ALGORITMOS
}

alvo_real_teste2024 = predicoes_teste2024[ALGORITMOS[0]]["obito"].to_numpy()
for algoritmo in ALGORITMOS:
    assert np.array_equal(predicoes_teste2024[algoritmo]["obito"].to_numpy(), alvo_real_teste2024)

numero_registros_teste2024 = len(alvo_real_teste2024)
numero_registros_teste2024, alvo_real_teste2024.sum()

(3978, np.int64(208))

Recalcula AUPRC, ROC-AUC e Brier Score de cada algoritmo sobre o teste de 2024 completo, usando `avaliar_predicoes` de `utils/preprocessamento.py`, como estimativa pontual de referencia para as secoes seguintes.

In [3]:
metricas_pontuais = []
for algoritmo in ALGORITMOS:
    probabilidade_predita = predicoes_teste2024[algoritmo]["probabilidade_predita"].to_numpy()
    metricas = preprocessamento.avaliar_predicoes(alvo_real_teste2024, probabilidade_predita)
    metricas["algoritmo"] = algoritmo
    metricas_pontuais.append(metricas)

tabela_metricas_pontuais = pd.DataFrame(metricas_pontuais).set_index("algoritmo")
tabela_metricas_pontuais.sort_values("auprc", ascending=False)

,auprc,roc_auc,brier_score
algoritmo,,,
random_forest,0.227791,0.816974,0.066202
regressao_logistica,0.207991,0.818218,0.236374
lightgbm,0.192526,0.768216,0.155528
xgboost,0.133091,0.685565,0.072707
arvore_decisao,0.120056,0.731971,0.225455


Gera, uma unica vez, os indices de reamostragem com reposicao usados em todas as repeticoes de bootstrap.
Usar os mesmos indices de reamostragem para os cinco algoritmos garante que a comparacao pareada entre algoritmos, feita mais adiante, seja valida, ja que cada reamostragem representa o mesmo conjunto hipotetico de casos para todos os algoritmos.
Uma semente fixa garante que os resultados sejam reproduziveis entre execucoes do notebook.

In [4]:
gerador_aleatorio = np.random.default_rng(SEMENTE_ALEATORIA)

indices_reamostragem = gerador_aleatorio.integers(
    low=0, high=numero_registros_teste2024, size=(NUMERO_REAMOSTRAGENS, numero_registros_teste2024)
)

indices_reamostragem.shape

(2000, 3978)

Para cada algoritmo e cada reamostragem, recalcula AUPRC e ROC-AUC sobre a amostra reamostrada do teste de 2024.
Uma reamostragem que, por acaso, nao contenha nenhum obito e descartada dessa metrica especifica, ja que AUPRC e ROC-AUC nao sao definidos sem as duas classes presentes, o que e raro dado que o teste de 2024 tem 208 obitos, mas e tratado de forma explicita por seguranca.
O resultado e uma distribuicao bootstrap de AUPRC e de ROC-AUC para cada algoritmo, guardada em um dicionario indexado pelo nome do algoritmo.

In [5]:
from sklearn.metrics import average_precision_score, roc_auc_score

distribuicao_bootstrap_auprc = {}
distribuicao_bootstrap_roc_auc = {}

for algoritmo in ALGORITMOS:
    probabilidade_predita = predicoes_teste2024[algoritmo]["probabilidade_predita"].to_numpy()

    valores_auprc = []
    valores_roc_auc = []
    for linha_indices in indices_reamostragem:
        alvo_reamostrado = alvo_real_teste2024[linha_indices]
        probabilidade_reamostrada = probabilidade_predita[linha_indices]
        if alvo_reamostrado.sum() == 0 or alvo_reamostrado.sum() == len(alvo_reamostrado):
            continue
        valores_auprc.append(average_precision_score(alvo_reamostrado, probabilidade_reamostrada))
        valores_roc_auc.append(roc_auc_score(alvo_reamostrado, probabilidade_reamostrada))

    distribuicao_bootstrap_auprc[algoritmo] = np.array(valores_auprc)
    distribuicao_bootstrap_roc_auc[algoritmo] = np.array(valores_roc_auc)

{algoritmo: len(distribuicao_bootstrap_auprc[algoritmo]) for algoritmo in ALGORITMOS}

{'regressao_logistica': 2000,
 'arvore_decisao': 2000,
 'random_forest': 2000,
 'xgboost': 2000,
 'lightgbm': 2000}

Constroi a tabela final de intervalos de confianca de 95 por cento para AUPRC e ROC-AUC de cada algoritmo, usando os percentis 2,5 e 97,5 das respectivas distribuicoes bootstrap.
A estimativa pontual reportada e a calculada sobre o teste de 2024 completo, e nao a media da distribuicao bootstrap, para manter coerencia com a tabela comparativa produzida na fase de modelagem.

In [6]:
linhas_intervalo_confianca = []
for algoritmo in ALGORITMOS:
    for nome_metrica, distribuicao_bootstrap in [
        ("auprc", distribuicao_bootstrap_auprc[algoritmo]),
        ("roc_auc", distribuicao_bootstrap_roc_auc[algoritmo]),
    ]:
        linhas_intervalo_confianca.append(
            {
                "algoritmo": algoritmo,
                "metrica": nome_metrica,
                "estimativa_pontual": tabela_metricas_pontuais.loc[algoritmo, nome_metrica],
                "limite_inferior_ic95": np.percentile(distribuicao_bootstrap, 2.5),
                "limite_superior_ic95": np.percentile(distribuicao_bootstrap, 97.5),
            }
        )

tabela_intervalo_confianca = pd.DataFrame(linhas_intervalo_confianca)
tabela_intervalo_confianca.sort_values(["metrica", "estimativa_pontual"], ascending=[True, False])

,algoritmo,metrica,estimativa_pontual,limite_inferior_ic95,limite_superior_ic95
4,random_forest,auprc,0.227791,0.181809,0.289026
0,regressao_logistica,auprc,0.207991,0.167899,0.261085
8,lightgbm,auprc,0.192526,0.152059,0.246635
6,xgboost,auprc,0.133091,0.099946,0.178547
2,arvore_decisao,auprc,0.120056,0.099087,0.142997
1,regressao_logistica,roc_auc,0.818218,0.788278,0.845471
5,random_forest,roc_auc,0.816974,0.787600,0.844707
9,lightgbm,roc_auc,0.768216,0.734440,0.800286
3,arvore_decisao,roc_auc,0.731971,0.699229,0.767495
7,xgboost,roc_auc,0.685565,0.647570,0.721997


Identifica os dois algoritmos com maior AUPRC pontual no teste de 2024, para a comparacao pareada por bootstrap feita a seguir.

In [7]:
ranking_auprc = tabela_metricas_pontuais["auprc"].sort_values(ascending=False)
melhor_algoritmo, segundo_melhor_algoritmo = ranking_auprc.index[0], ranking_auprc.index[1]

melhor_algoritmo, segundo_melhor_algoritmo

('random_forest', 'regressao_logistica')

Para cada uma das mesmas reamostragens usadas acima, calcula a diferenca de AUPRC entre o melhor e o segundo melhor algoritmo.
Usar a mesma reamostragem para os dois algoritmos, em vez de reamostrar cada um de forma independente, e o que torna essa comparacao pareada, e reduz a variancia da diferenca estimada em relacao a comparar as duas distribuicoes bootstrap de forma nao pareada.
Reamostragens sem nenhum obito ou sem nenhum sobrevivente sao descartadas, pela mesma razao da secao anterior.

In [8]:
probabilidade_melhor = predicoes_teste2024[melhor_algoritmo]["probabilidade_predita"].to_numpy()
probabilidade_segundo_melhor = predicoes_teste2024[segundo_melhor_algoritmo]["probabilidade_predita"].to_numpy()

diferencas_auprc_pareadas = []
for linha_indices in indices_reamostragem:
    alvo_reamostrado = alvo_real_teste2024[linha_indices]
    if alvo_reamostrado.sum() == 0 or alvo_reamostrado.sum() == len(alvo_reamostrado):
        continue
    auprc_melhor = average_precision_score(alvo_reamostrado, probabilidade_melhor[linha_indices])
    auprc_segundo_melhor = average_precision_score(alvo_reamostrado, probabilidade_segundo_melhor[linha_indices])
    diferencas_auprc_pareadas.append(auprc_melhor - auprc_segundo_melhor)

diferencas_auprc_pareadas = np.array(diferencas_auprc_pareadas)

diferenca_pontual_auprc = (
    tabela_metricas_pontuais.loc[melhor_algoritmo, "auprc"]
    - tabela_metricas_pontuais.loc[segundo_melhor_algoritmo, "auprc"]
)
fracao_reamostragens_com_diferenca_positiva = (diferencas_auprc_pareadas > 0).mean()

linha_comparacao_pareada = {
    "algoritmo": f"{melhor_algoritmo}_menos_{segundo_melhor_algoritmo}",
    "metrica": "diferenca_auprc",
    "estimativa_pontual": diferenca_pontual_auprc,
    "limite_inferior_ic95": np.percentile(diferencas_auprc_pareadas, 2.5),
    "limite_superior_ic95": np.percentile(diferencas_auprc_pareadas, 97.5),
}

linha_comparacao_pareada, fracao_reamostragens_com_diferenca_positiva

({'algoritmo': 'random_forest_menos_regressao_logistica',
  'metrica': 'diferenca_auprc',
  'estimativa_pontual': np.float64(0.019799281118263096),
  'limite_inferior_ic95': np.float64(-0.02175297757034811),
  'limite_superior_ic95': np.float64(0.06465240657727415)},
 np.float64(0.8365))

Reune os intervalos de confianca por algoritmo e a comparacao pareada numa unica tabela final, e salva o resultado em `analysis/results_avaliacao/comparacao_bootstrap_teste2024.csv`.
A fracao de reamostragens com diferenca positiva entre os dois melhores algoritmos e registrada como coluna adicional, apenas na linha da comparacao pareada, como uma leitura direta da forca da evidencia a favor do melhor algoritmo.

In [9]:
tabela_intervalo_confianca["fracao_reamostragens_diferenca_positiva"] = np.nan
linha_comparacao_pareada["fracao_reamostragens_diferenca_positiva"] = fracao_reamostragens_com_diferenca_positiva

tabela_final_bootstrap = pd.concat(
    [tabela_intervalo_confianca, pd.DataFrame([linha_comparacao_pareada])], ignore_index=True
)

tabela_final_bootstrap.to_csv(
    DIRETORIO_RESULTADOS_ANALISE / "comparacao_bootstrap_teste2024.csv", index=False
)
tabela_final_bootstrap

,algoritmo,metrica,estimativa_pontual,limite_inferior_ic95,limite_superior_ic95,fracao_reamostragens_diferenca_positiva
0,regressao_logistica,auprc,0.207991,0.167899,0.261085,NaN
1,regressao_logistica,roc_auc,0.818218,0.788278,0.845471,NaN
2,arvore_decisao,auprc,0.120056,0.099087,0.142997,NaN
3,arvore_decisao,roc_auc,0.731971,0.699229,0.767495,NaN
4,random_forest,auprc,0.227791,0.181809,0.289026,NaN
5,random_forest,roc_auc,0.816974,0.787600,0.844707,NaN
6,xgboost,auprc,0.133091,0.099946,0.178547,NaN
7,xgboost,roc_auc,0.685565,0.647570,0.721997,NaN
8,lightgbm,auprc,0.192526,0.152059,0.246635,NaN
9,lightgbm,roc_auc,0.768216,0.734440,0.800286,NaN


## Sintese dos achados

Os intervalos de confianca de 95 por cento de AUPRC se sobrepoem entre a maioria dos cinco algoritmos, o que reflete diretamente o numero reduzido de obitos no teste de 2024, apenas 208 entre 3978 registros.
Random Forest, Regressao Logistica e LightGBM formam um grupo com os intervalos de AUPRC mais altos e mutuamente sobrepostos, de 0,182 a 0,289, de 0,168 a 0,261 e de 0,152 a 0,247 respectivamente, enquanto XGBoost e Arvore de Decisao formam um segundo grupo com intervalos mais baixos e quase totalmente sobrepostos entre si, de 0,100 a 0,179 e de 0,099 a 0,143 respectivamente.
O intervalo do Random Forest, o mais alto dos cinco, e o unico que nao se sobrepoe ao de nenhum dos dois algoritmos do grupo mais baixo, enquanto os intervalos da Regressao Logistica e do LightGBM ainda se sobrepoem parcialmente ao intervalo do XGBoost, o melhor dos dois algoritmos do grupo mais baixo.
A comparacao pareada por bootstrap entre Random Forest e Regressao Logistica, os dois algoritmos com maior AUPRC pontual, mostra uma diferenca pontual de 0,0198 a favor do Random Forest, com intervalo de confianca de 95 por cento entre -0,022 e 0,065, que ainda inclui o zero, mas a diferenca e positiva em 83,65 por cento das reamostragens, uma proporcao bem acima dos 50 por cento esperados sob ausencia de qualquer vantagem real.
Essa leitura sugere uma tendencia mais consistente a favor do Random Forest do que uma diferenca puramente ao acaso, ainda que sem atingir o limiar convencional de significancia estatistica de 95 por cento, e a escolha final entre os dois deve continuar se apoiando tambem em outros criterios, como a calibracao das probabilidades e a estabilidade entre 2024 e 2025, avaliados nos dois notebooks seguintes desta fase.
Pelo ROC-AUC, o XGBoost passa a ter o intervalo mais baixo entre os cinco algoritmos, de 0,648 a 0,722, abaixo ate do intervalo da Arvore de Decisao, de 0,699 a 0,767, o que reforca que o XGBoost e o algoritmo cujo desempenho mais se deteriorou nesta avaliacao.
XGBoost e Arvore de Decisao, portanto, sao os dois algoritmos que podem ser descartados com mais seguranca entre os cinco avaliados nesta comparacao.